# felix_06 — progress report for one or more FelixGenome runs

Give this one or more workflow output roots and it reports, per phenotype, how far
each testing arm has got. It **lists only** — no result data is downloaded, so it is
cheap to re-run while a workflow is going.

### Three arms, four tests

The workflow has three per-phenotype step-2 scatters, and they are not the four cells
of the 2×2:

| scatter | coding | what its TSV contains |
| --- | --- | --- |
| `RuStep2` | repeat dosage (`coding = "ru"`) | **both** `p.value_ancALL` (ancestry-collapsed) and `P_cct_admixed_c` (ancestry-informed) |
| `BaStep2` | carrier / biallelic (`coding = "carrier"`) | **both**, same two columns |
| `Step2` | — | the ordinary per-VCF-record scan: one marker per record, not per catalog locus |

So each repeat arm delivers **two** of the four 2×2 cells at once, and the 2×2 is
complete for a phenotype as soon as `RuStep2` and `BaStep2` both have it:

| | ancestry-collapsed (`p.value_ancALL`) | ancestry-informed (`P_cct_admixed_c`) |
| --- | --- | --- |
| **carrier coding** | A — conventional GWAS baseline | B |
| **repeat dosage** | C | D — production arm |

`Step2` is a different marker set and does not join to the others, so it is reported
separately rather than as a fourth cell.

### Notes

- Shards are found by listing each root recursively and matching the call name at any
  depth, because Cromwell compiles the three step-2 scatters into anonymous
  `call-ScatterAt<line>_<col>/` nodes. `scripts/felix_interim_scan.sh` owns that logic
  (and the highest-attempt rule); this notebook drives it rather than reimplementing it.
- Several roots can hold the shards of one logical run: a resubmission that call-cached
  a task does not necessarily recreate its directory. Pass every root — shards are
  unioned, and the last root wins a shared shard.
- One recursive listing per root, cached in `LISTING_DIR` and reused by all three arms
  and by the upstream-stage report.

| Variable | Default | What it does |
| --- | --- | --- |
| `FELIX_RUNS` | — | comma-separated `gs://` workflow roots (**the main input**) |
| `FELIX_SUBMISSION_ID` + `FELIX_WORKFLOW_ID` | — | used to build one root if `FELIX_RUNS` is unset |
| `FELIX_PHENOTYPES` | `$WORKSPACE_BUCKET/tractor_mix_pilot/selected_phenotypes.txt` | scatter order; the inner shard index is the line number |
| `FELIX_MIN_CHROMS` | `22` | chromosomes required to call a phenotype complete |
| `FELIX_PROGRESS_DIR` | `felix_progress` | output directory |
| `FELIX_RELIST` | `0` | `1` discards the cached listings and re-lists |

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError("scripts/ not found locally and WORKSPACE_BUCKET is unset.")
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"])
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook("felix_interim_scan.sh")


def sh(cmd: str, check: bool = True) -> int:
    '''Run a shell command and fail on a non-zero exit.

    get_ipython().system() returns None even when the command fails, so the
    usual `if rc: raise` never fires.
    '''
    print(cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


WS = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
assert WS, "WORKSPACE_BUCKET is unset"

INTERIM = SCRIPTS / "felix_interim_scan.sh"
assert INTERIM.is_file(), INTERIM
print("interim scan :", INTERIM)

In [ ]:
RUNS = [r.strip().rstrip("/") for r in os.environ.get("FELIX_RUNS", "").split(",") if r.strip()]
if not RUNS:
    sid, wid = os.environ.get("FELIX_SUBMISSION_ID", ""), os.environ.get("FELIX_WORKFLOW_ID", "")
    name = os.environ.get("WORKFLOW_NAME", "FelixGenome")
    if not (sid and wid):
        raise SystemExit("Set FELIX_RUNS to one or more gs:// workflow roots, "
                         "or FELIX_SUBMISSION_ID + FELIX_WORKFLOW_ID.")
    RUNS = [f"{WS}/submissions/{sid}/{name}/{wid}"]

PHENOTYPES = os.environ.get("FELIX_PHENOTYPES", f"{WS}/tractor_mix_pilot/selected_phenotypes.txt")
MIN_CHROMS = int(os.environ.get("FELIX_MIN_CHROMS", "22"))
WORK = Path(os.environ.get("FELIX_PROGRESS_DIR", "felix_progress"))
WORK.mkdir(parents=True, exist_ok=True)
LISTING_DIR = WORK / "listings"
LISTING_DIR.mkdir(parents=True, exist_ok=True)

if os.environ.get("FELIX_RELIST", "0") == "1":
    for _f in LISTING_DIR.glob("listing-*.txt"):
        _f.unlink()
    print("discarded cached listings")

RUN_ARGS = " ".join(f"--run {r}" for r in RUNS)
ARMS = ["main", "ru", "ru_biallelic"]
ARM_LABEL = {"main": "Step2 (ordinary, per-record)",
             "ru": "RuStep2 (repeat dosage)",
             "ru_biallelic": "BaStep2 (carrier/biallelic)"}

for i, r in enumerate(RUNS, 1):
    print(f"run {i}: {r}")
print("phenos :", PHENOTYPES)
print("work   :", WORK.resolve())
print("complete at >=", MIN_CHROMS, "chromosomes")

## Per-arm coverage

`--coverage-only` lists and counts; it merges nothing and downloads no result data. An
arm with no shards yet exits non-zero and is reported as absent rather than failing the
notebook.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

cov = {}
for arm in ARMS:
    out = WORK / f"progress_{arm}"
    rc = sh(f"bash {INTERIM} {RUN_ARGS} --phenotypes {PHENOTYPES} "
            f"--arm {arm} --min-chroms {MIN_CHROMS} --out {out} "
            f"--listing-dir {LISTING_DIR} --coverage-only", check=False)
    f = out / "coverage.tsv"
    if f.is_file():
        cov[arm] = pd.read_csv(f, sep="\t")
    else:
        print(f"  arm {arm}: no step-2 shards found yet (exit {rc})")

if not cov:
    raise SystemExit("No step-2 shards found in any arm, in any run root given.")

PHENO_ORDER = cov[next(iter(cov))]["phenotype"].tolist()
NPHENO = len(PHENO_ORDER)
print(f"\n{NPHENO} phenotypes in scatter order")

In [ ]:
chrom = pd.DataFrame(index=PHENO_ORDER)
for arm in ARMS:
    if arm in cov:
        chrom[arm] = cov[arm].set_index("phenotype")["chromosomes_present"]
    else:
        chrom[arm] = 0
chrom = chrom.fillna(0).astype(int)
chrom.index.name = "phenotype"

over = chrom[(chrom[ARMS] > 22).any(axis=1)]
if len(over):
    print("!! more than 22 chromosomes for some phenotype/arm. The same logical shard\n"
          "   is being counted under more than one run root (differing sub-workflow\n"
          "   uuids), so these counts are inflated:")
    print(over.to_string())

summary = pd.DataFrame({
    "arm": [ARM_LABEL[a] for a in ARMS],
    "shards_present": [int(chrom[a].sum()) for a in ARMS],
    "shards_expected": [NPHENO * 22] * len(ARMS),
    "phenotypes_started": [int((chrom[a] > 0).sum()) for a in ARMS],
    f"phenotypes_complete_{MIN_CHROMS}": [int((chrom[a] >= MIN_CHROMS).sum()) for a in ARMS],
    "max_chroms_any_pheno": [int(chrom[a].max()) for a in ARMS],
}, index=ARMS)
summary["pct_shards"] = (100 * summary["shards_present"] / summary["shards_expected"]).round(1)
summary.to_csv(WORK / "progress_summary.tsv", sep="\t")
chrom.to_csv(WORK / "progress_chromosomes.tsv", sep="\t")
summary[["arm", "shards_present", "shards_expected", "pct_shards",
         "phenotypes_started", f"phenotypes_complete_{MIN_CHROMS}", "max_chroms_any_pheno"]]

## Which of the four tests each phenotype has

A phenotype's dosage pair (C and D) comes from `RuStep2`, its carrier pair (A and B)
from `BaStep2`. Both arms at `MIN_CHROMS` means the full 2×2 is available for that
phenotype; one arm means a half-grid, which supports the local-ancestry contrast within
that coding but not the coding contrast.

In [ ]:
ru_ok = chrom["ru"] >= MIN_CHROMS
ba_ok = chrom["ru_biallelic"] >= MIN_CHROMS

tests = pd.DataFrame({
    "A_pooled_carrier": ba_ok, "B_ancestry_carrier": ba_ok,
    "C_pooled_dosage": ru_ok, "D_ancestry_dosage": ru_ok,
    "ordinary_scan": chrom["main"] >= MIN_CHROMS,
})
tests.index.name = "phenotype"


def grid_state(row):
    if row["C_pooled_dosage"] and row["A_pooled_carrier"]:
        return "full 2x2"
    if row["C_pooled_dosage"]:
        return "dosage pair only (C, D)"
    if row["A_pooled_carrier"]:
        return "carrier pair only (A, B)"
    return "neither repeat arm complete"


tests["grid"] = tests.apply(grid_state, axis=1)
tests.to_csv(WORK / "progress_tests.tsv", sep="\t")

counts = tests["grid"].value_counts().rename("phenotypes").to_frame()
counts["of_total"] = (100 * counts["phenotypes"] / NPHENO).round(1)
print(f"2x2 availability at >= {MIN_CHROMS} chromosomes per arm, {NPHENO} phenotypes:\n")
print(counts.to_string())
print("\nordinary per-record scan complete for",
      int(tests["ordinary_scan"].sum()), "phenotypes")

In [ ]:
# Phenotypes that are lagging, longest tail first. Partial phenotypes are the
# interesting ones: a repeat arm below MIN_CHROMS is a chromosome subset, and
# anything computed from it must be labelled that way.
lag = chrom[(chrom["ru"] < MIN_CHROMS) | (chrom["ru_biallelic"] < MIN_CHROMS)]
lag = lag.assign(grid=tests.loc[lag.index, "grid"]).sort_values(["ru", "ru_biallelic"])
print(f"{len(lag)} of {NPHENO} phenotypes are short of {MIN_CHROMS} chromosomes in at "
      f"least one repeat arm.")
lag.head(30)

## Upstream stages

The repeat arms cannot start a chromosome until `RuLoci` and then `RuVcf` / `BaVcf`
finish it, so a lagging repeat arm is usually an upstream gap rather than a step-2
problem. This reads the **cached** listings — no extra bucket traffic.

`executed_rc` counts shards that wrote Cromwell's `rc` file, i.e. that actually ran the
command here (successfully or not). A **call-cache hit never writes an `rc`** — Cromwell
copies the outputs instead — so `no_rc` (objects present, no `rc`) is the cache-hit count,
derived rather than assumed. `cache_copied` additionally counts shards whose outputs sit
under a `cacheCopy/` directory; if `no_rc` is large while `cache_copied` is 0, the outputs
arrived by some other route and the Submission History "Call Caching Result" column is the
place to confirm.

A task with **no row at all** has not produced a single object: nothing dispatched. That is
what a scatter full of `—` shards looks like from the bucket.

In [ ]:
import re
from collections import defaultdict

SEG_RE = re.compile(r"^(call-|shard-)")
seen = defaultdict(set)       # task -> shards with any object
executed = defaultdict(set)   # task -> shards that wrote an rc (ran a command)
copied = defaultdict(set)     # task -> shards with a cacheCopy/ segment
listed = 0
nlist = len(list(LISTING_DIR.glob("listing-*.txt")))

for lf in sorted(LISTING_DIR.glob("listing-*.txt")):
    with lf.open() as fh:
        for line in fh:
            line = line.rstrip("\n")
            listed += 1
            parts = line.split("/")
            # Identify a shard by its call/shard skeleton: the sequence of
            # call-<Name> and shard-<N> segments, dropping attempt-<K>, the
            # bare ScatterAt<line>_<col> repeat and the sub-workflow uuid.
            # Those last two differ between submissions for the SAME logical
            # shard, so a root-relative path would double-count a shard that
            # appears under both an original run and its rescue.
            segs = [p for p in parts[:-1] if SEG_RE.match(p)]
            calls = [p[5:] for p in segs if p.startswith("call-")]
            if not calls:
                continue
            # Innermost call- segment is the real task; outer ones are the
            # anonymous ScatterAt wrappers Cromwell compiles the scatters into.
            task, key = calls[-1], "/".join(segs)
            seen[task].add(key)
            if parts[-1] == "rc":
                executed[task].add(key)
            if "cacheCopy" in parts:
                copied[task].add(key)

stage = pd.DataFrame({
    "shards_with_objects": {k: len(v) for k, v in seen.items()},
    "executed_rc": {k: len(v) for k, v in executed.items()},
    "cache_copied": {k: len(v) for k, v in copied.items()},
}).fillna(0).astype(int)
# A shard with objects but no rc did not run a command here. That is the
# signature of a call-cache hit, whether or not Cromwell wrote the outputs
# under a cacheCopy/ directory -- so it is derived rather than assumed.
stage["no_rc"] = stage["shards_with_objects"] - stage["executed_rc"]
stage = stage.sort_values("shards_with_objects", ascending=False)
stage.index.name = "task"
stage.to_csv(WORK / "progress_stages.tsv", sep="\t")
print(f"{listed:,} objects across {nlist} cached listing(s)\n")
stage

## Progress figure

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm

ARM_COLOR = {"main": "#8c8c8c", "ru": "#4C72B0", "ru_biallelic": "#DD8452"}
ARM_SHORT = {"main": "ordinary", "ru": "dosage", "ru_biallelic": "carrier"}
order = chrom.sort_values(["ru", "ru_biallelic", "main"], ascending=False).index
mat = chrom.loc[order, ARMS].to_numpy(dtype=float)

h = max(2.6, 0.13 * len(order) + 1.5)
fig, (axh, axb) = plt.subplots(
    1, 2, figsize=(7.2, h), gridspec_kw={"width_ratios": [2.4, 1.0], "wspace": 0.45})

im = axh.imshow(mat, aspect="auto", cmap="viridis", vmin=0, vmax=22,
                interpolation="nearest")
axh.set_xticks(range(len(ARMS)))
axh.set_xticklabels([ARM_SHORT[a] for a in ARMS], fontsize=8)
for t, a in zip(axh.get_xticklabels(), ARMS):
    t.set_color(ARM_COLOR[a])
axh.set_yticks(range(len(order)))
axh.set_yticklabels(order, fontsize=5 if len(order) > 40 else 7)
axh.set_title(f"Chromosomes present per phenotype ({len(order)} phenotypes)",
              fontsize=9, loc="left")
for s in axh.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=axh, fraction=0.035, pad=0.02, ticks=[0, 11, 22])
cb.set_label("chromosomes of 22", fontsize=7)
cb.ax.tick_params(labelsize=7)

pct = summary["pct_shards"].to_numpy()
ypos = np.arange(len(ARMS))[::-1]
axb.barh(ypos, pct, color=[ARM_COLOR[a] for a in ARMS], height=0.6)
for y, p, n, e in zip(ypos, pct, summary["shards_present"], summary["shards_expected"]):
    axb.text(min(p + 2, 88), y, f"{p:.0f}%  ({n:,}/{e:,})", va="center", fontsize=7)
axb.set_yticks(ypos)
axb.set_yticklabels([ARM_SHORT[a] for a in ARMS], fontsize=8)
for t, a in zip(axb.get_yticklabels(), ARMS):
    t.set_color(ARM_COLOR[a])
axb.set_xlim(0, 118)
axb.set_xticks([0, 50, 100])
axb.set_xlabel("step-2 shards finished (%)", fontsize=8)
axb.set_title("Shard-level progress", fontsize=9, loc="left")
axb.tick_params(labelsize=7)
for s in ("top", "right"):
    axb.spines[s].set_visible(False)

fig.savefig(WORK / "progress.png", dpi=200, bbox_inches="tight")
print("wrote", (WORK / "progress.png").resolve())
fig

## How to read this

- **`pct_shards`** is the honest completion number: phenotypes × 22 chromosomes per arm.
  Phenotype counts move in steps of 22 shards and lag the real progress.
- **A repeat arm below 22 chromosomes is a chromosome subset.** λGC, Manhattan plots and
  the 2×2 contrasts computed from it are valid as paired comparisons at matched loci, but
  are not genome-wide numbers and significance *counts* are not interpretable at all.
- **A lagging repeat arm is usually upstream.** Check the stage table: `RuStep2` for
  chromosome *c* cannot start until `RuVcf` for *c* is finished, and `RuVcf` waits on
  `RuLoci`.
- **The ordinary arm is not the fourth cell of the grid.** It tests one marker per VCF
  record against the repeat arms' one marker per catalog locus; the marker sets do not
  join, so "did the repeat branch beat the ordinary branch" is not a p-value comparison
  between them.
- To go from progress to results, `felix_04_interim_scan.ipynb` merges whatever is
  present and plots it.

Files written into `FELIX_PROGRESS_DIR`:

| File | Contents |
| --- | --- |
| `progress_summary.tsv` | one row per arm: shards present/expected, phenotypes started/complete |
| `progress_chromosomes.tsv` | phenotype × arm chromosome counts |
| `progress_tests.tsv` | per phenotype, which of A/B/C/D and the ordinary scan are available |
| `progress_stages.tsv` | finished shards per Cromwell task |
| `progress.png` | the figure above |
| `listings/listing-*.txt` | cached recursive listing, one per run root |